In [2]:
from datasets import load_from_disk

ds = load_from_disk("../01_load_dataset/musiccaps/dataset_audio")
sample = ds[0]

In [3]:
from transformers import ClapProcessor, ClapModel
import torch
import librosa
import numpy as np

processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").eval()

# Get audio from dataset
audio_data = ds[0]["audio"]["array"]
original_sr = ds[0]["audio"]["sampling_rate"]  # 44100

# Resample to 48000 if needed
if original_sr != 48000:
    audio_data = librosa.resample(
        audio_data, 
        orig_sr=original_sr, 
        target_sr=48000
    )

# Process with CLAP
inputs = processor(
    audios=audio_data,
    sampling_rate=48000,
    return_tensors="pt"
)

with torch.no_grad():
    audio_emb = clap.get_audio_features(**inputs)

print(audio_emb.shape)

/Users/aliozkaya/miniconda3/envs/audio-caption/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


torch.Size([1, 512])


In [4]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2.train()


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

/Users/aliozkaya/miniconda3/envs/audio-caption/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
D_AUDIO = audio_emb.size(-1)          # e.g. 512
D_LM = gpt2.config.n_embd             # e.g. 768
PREFIX_LEN = 10

projection = nn.Linear(D_AUDIO, PREFIX_LEN * D_LM).to(device)


In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Move model to device
projection = projection.to(device)

# Move input to device before forward pass
audio_emb = audio_emb.to(device)
prefix = projection(audio_emb)
prefix = prefix.view(-1, PREFIX_LEN, D_LM)

In [8]:
captions = ["a slow piano melody with strings"]  # batch of strings

tokens = tokenizer(
    captions,
    return_tensors="pt",
    padding=True
).to(device)

input_ids = tokens.input_ids                    # (B, T)


In [9]:
text_embeds = gpt2.transformer.wte(input_ids)   # (B, T, D_LM)


In [10]:
inputs_embeds = torch.cat([prefix, text_embeds], dim=1)


In [11]:
labels = input_ids.clone()

# pad labels to match prefix length
prefix_labels = torch.full(
    (labels.size(0), PREFIX_LEN),
    -100,                          # ignore index
    device=device
)

labels = torch.cat([prefix_labels, labels], dim=1)


In [12]:
outputs = gpt2(
    inputs_embeds=inputs_embeds,
    labels=labels
)

loss = outputs.loss
loss.backward()
